# Solutions — Wanderbricks Payments (Medium 07)

**Dataset:** `samples.wanderbricks.payments`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

payments = spark.table("samples.wanderbricks.payments")

## Solution 1 — Revenue per Payment Method (Completed Only)

In [ ]:
result_1 = (
    payments
    .filter(F.col("status") == "completed")
    .groupBy("payment_method")
    .agg(
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.count("*").alias("payment_count")
    )
    .orderBy(F.col("total_amount").desc())
)

## Solution 2 — Failed Payment Rate per Method

In [ ]:
result_2 = (
    payments
    .groupBy("payment_method")
    .agg(
        F.count("*").alias("total_payments"),
        F.sum(F.when(F.col("status") == "failed", 1).otherwise(0)).alias("failed_payments")
    )
    .withColumn("failure_rate_pct", F.round(F.col("failed_payments") / F.col("total_payments") * 100, 2))
    .orderBy(F.col("failure_rate_pct").desc())
)

## Solution 3 — Monthly Payment Volume

In [ ]:
result_3 = (
    payments
    .withColumn("year", F.year("payment_date"))
    .withColumn("month", F.month("payment_date"))
    .groupBy("year","month")
    .agg(
        F.count("*").alias("payment_count"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
    .orderBy("year","month")
)

## Solution 4 — Running Total by Payment Date

In [ ]:
w = Window.orderBy("payment_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result_4 = (
    payments
    .select(
        "payment_id","payment_date","amount",
        F.round(F.sum("amount").over(w), 2).alias("running_total")
    )
    .orderBy("payment_date")
)

## Solution 5 — Bookings with More Than 1 Payment

In [ ]:
result_5 = (
    payments
    .groupBy("booking_id")
    .agg(
        F.count("*").alias("payment_count"),
        F.round(F.sum("amount"), 2).alias("total_paid")
    )
    .filter(F.col("payment_count") > 1)
    .orderBy(F.col("payment_count").desc())
)

## Solution 6 — Revenue Percentage per Method (Completed)

In [ ]:
w = Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
result_6 = (
    payments
    .filter(F.col("status") == "completed")
    .groupBy("payment_method")
    .agg(F.round(F.sum("amount"), 2).alias("total_amount"))
    .withColumn(
        "revenue_pct",
        F.round(F.col("total_amount") / F.sum("total_amount").over(w) * 100, 2)
    )
    .orderBy(F.col("revenue_pct").desc())
)

## Solution 7 — Avg Days Between Payments per Booking (Lag)

In [ ]:
# Why: lag gets previous payment date within same booking; datediff computes the gap
w = Window.partitionBy("booking_id").orderBy("payment_date")
result_7 = (
    payments
    .withColumn("prev_payment_date", F.lag("payment_date", 1).over(w))
    .withColumn("days_between", F.datediff(F.col("payment_date"), F.col("prev_payment_date")))
    .filter(F.col("days_between").isNotNull())
    .select("booking_id","payment_id","payment_date","prev_payment_date","days_between")
    .orderBy("booking_id","payment_date")
)